# Synthetic Financial Stress Outcome Validation

This notebook validates the synthetic `financial_stress_event` outcome. It does not train a machine-learning model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.alternative_features import FEATURE_COLUMNS
from src.features.financial_stress_outcome import save_model_dataset, validate_model_dataset

sns.set_theme(style='whitegrid')

In [ ]:
model_path = PROJECT_ROOT / 'data' / 'processed' / 'synthetic' / 'model_dataset.csv'
profile_path = PROJECT_ROOT / 'data' / 'raw' / 'synthetic' / 'applicant_profiles.csv'

if not model_path.exists():
    model_dataset = save_model_dataset(PROJECT_ROOT, seed=42)
else:
    model_dataset = pd.read_csv(model_path)

profiles = pd.read_csv(profile_path)
validate_model_dataset(model_dataset)

analysis = profiles[['applicant_id', 'population_group']].merge(model_dataset, on='applicant_id', how='inner')
model_dataset.shape, analysis.shape

In [ ]:
event_rate = model_dataset['financial_stress_event'].mean()
prob_event_corr = model_dataset['financial_stress_probability'].corr(model_dataset['financial_stress_event'])

pd.Series({
    'overall_event_rate': event_rate,
    'probability_event_correlation': prob_event_corr,
    'minimum_probability': model_dataset['financial_stress_probability'].min(),
    'maximum_probability': model_dataset['financial_stress_probability'].max(),
}).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(model_dataset['financial_stress_probability'], bins=50, ax=axes[0])
axes[0].set_title('Financial Stress Probability')
sns.boxplot(data=analysis, x='population_group', y='financial_stress_probability', ax=axes[1])
axes[1].set_title('Probability by Synthetic Population')
fig.tight_layout()

In [ ]:
analysis.groupby('population_group').agg(
    event_rate=('financial_stress_event', 'mean'),
    mean_probability=('financial_stress_probability', 'mean'),
    probability_std=('financial_stress_probability', 'std'),
    count=('applicant_id', 'count'),
).round(4)

In [ ]:
probability_correlations = model_dataset[FEATURE_COLUMNS].corrwith(model_dataset['financial_stress_probability']).sort_values(key=lambda s: s.abs(), ascending=False)
event_correlations = model_dataset[FEATURE_COLUMNS].corrwith(model_dataset['financial_stress_event']).sort_values(key=lambda s: s.abs(), ascending=False)

pd.DataFrame({
    'corr_with_probability': probability_correlations,
    'corr_with_event': event_correlations,
}).round(4)

In [ ]:
max_single_feature_event_corr = event_correlations.abs().max()
max_single_feature_probability_corr = probability_correlations.abs().max()

pd.Series({
    'max_abs_feature_corr_with_event': max_single_feature_event_corr,
    'max_abs_feature_corr_with_probability': max_single_feature_probability_corr,
}).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=analysis, x='financial_stress_probability', hue='population_group', common_norm=False, ax=axes[0])
axes[0].set_title('Population Overlap in Stress Probability')
sns.scatterplot(
    data=analysis.sample(n=min(1500, len(analysis)), random_state=42),
    x='expense_to_income_ratio',
    y='financial_buffer_ratio',
    hue='financial_stress_event',
    alpha=0.5,
    ax=axes[1],
)
axes[1].set_title('Outcome Is Stochastic Across Financial Profiles')
fig.tight_layout()

## Validation Notes

- The population group is used only for descriptive sanity checks in this notebook.
- The model dataset does not contain population labels, latent traits, or conventional credit-history variables.
- The outcome should show economic signal without becoming deterministic.
- This synthetic outcome is not loan default and must not be interpreted as real-world repayment behavior.